# Predicting Singapore gasoil, then estimating MGO

This notebook implements your team's two-stage approach in `docs/PRD.md`. First, multiple linear regression forecasts a gasoil return. Second, a proposed percentage-transfer rule converts that forecast into an MGO estimate using an observed MGO price anchor.

**This is an executed first experiment. It does not establish a validated forecast.** Source CSVs are unchanged. The initial horizon is the next observed gasoil trading date; confirm it with the team before treating it as a requirement.

Use **Run All** to reproduce the results. The numerical regression uses NumPy ordinary least squares; no scikit-learn installation is required. NumPy and pandas are the only analysis dependencies.


In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

# Works when Jupyter starts at the repository root or inside linear regression/.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "linear regression" / "gasoil_regression.py").exists():
    if (PROJECT_DIR / "gasoil_regression.py").exists():
        PROJECT_DIR = PROJECT_DIR.parent
    else:
        raise RuntimeError("Start this notebook from the repository root or linear regression directory")
sys.path.insert(0, str(PROJECT_DIR / "linear regression"))
from gasoil_regression import run

# Change this path on another computer.
DATA_DIR = Path("/Users/nelsontan/Downloads/market_data_cleaned")
OUTPUT_DIR = PROJECT_DIR / "linear regression" / "outputs"
result = run(DATA_DIR, OUTPUT_DIR)
print("Completed analysis. Outputs:", OUTPUT_DIR)
print("Verified market CSV SHA-256:", result["source_sha256"])


Completed analysis. Outputs: /Users/nelsontan/Desktop/Y2S1/CS203/cs203-g2t5/linear regression/outputs
Verified market CSV SHA-256: abbb18839f136aa94c413dc2bbe63a3f04b2b11233b9853fd5762093c97a7bbc


## 1. Check the data

Gasoil is the forecasting target; MGO quotes are anchors and observed outcomes for the second stage. Gas oil and crude are USD/barrel; MGO is USD/metric tonne; the broad USD index is an index, not an exchange rate. Returning percentage movements avoids equating these units, but does not prove that the products move together.

We use only the small market table. The large news evidence files are not loaded. News and monthly sales are outside this initial specification.


In [2]:
print(pd.DataFrame(result["coverage"]).to_string(index=False))
print("\nRow accounting:")
print(json.dumps(result["row_accounting"], indent=2))


  name                series  rows  observed  missing      first       last
gasoil   sg_gasoil_investing  2510      2510        0 2016-10-06 2026-10-05
 brent            brent_fred  2604      2531       73 2016-10-06 2026-09-29
 dubai dubai_crude_investing  2513      2513        0 2016-10-06 2026-10-05
   usd        usd_index_fred  2607      2492      115 2016-10-06 2026-10-02
   mgo    sg_mgo_themaritime   253       253        0 2020-03-13 2026-10-06

Row accounting:
{
  "gasoil_source_rows": 2510,
  "feature_candidates": 2509,
  "complete_rows": 2507,
  "boundary_purged": 2,
  "exclusions": {
    "": 2507,
    "missing_feature_or_target": 2
  },
  "adjacent_mgo_intervals": 252,
  "eligible_mapping_intervals": 115,
  "end_to_end_rows": 90
}


## 2. Define the linear regression

The target is `G_future / G_origin - 1`, a decimal return (0.01 means 1%). Inputs are gasoil, Brent, Dubai and USD index returns ending **no later than the previous gasoil trading date**. That extra lag is a conservative initial choice because publication times are unresolved. Missing source values are not filled; observed returns may span source gaps. Source dates, return starts and feature ages remain in the output CSV.

OLS fits `target_return = intercept + sum(coefficient × historical_return)`. Standardization uses training data only and is folded back into the printed original-scale coefficients. The coefficients describe associations, not economic causes.

Chronological split: approximately 70% train, 15% validation, 15% test. Rows crossing boundaries are removed. Coefficients stay frozen after training. Validation and test errors are measured on later observations.


In [3]:
model = result["gasoil_model"]
coefs = pd.DataFrame({"input": ["intercept"] + result["features"],
                      "coefficient": [model["intercept"]] + model["slopes"]})
print(coefs.to_string(index=False))
print("\nTraining observations:", model["training_rows"])
print("Training outcome cutoff:", result["train_end"])
print("Validation outcome cutoff:", result["validation_end"])
predictions = pd.read_csv(OUTPUT_DIR / "gasoil_predictions.csv")
print("\nExample held-out gasoil predictions:")
print(predictions.loc[predictions.split.eq("test"),
      ["origin", "target_date", "gasoil_origin", "gasoil_actual", "predicted_return", "gasoil_predicted"]]
      .head().to_string(index=False))


        input  coefficient
    intercept     0.000568
gasoil_return    -0.048487
 brent_return     0.023675
 dubai_return     0.022234
   usd_return     0.019399

Training observations: 1754
Training outcome cutoff: 2023-10-02
Validation outcome cutoff: 2025-04-01

Example held-out gasoil predictions:
    origin target_date  gasoil_origin  gasoil_actual  predicted_return  gasoil_predicted
2025-04-02  2025-04-03          89.09          84.75          0.000678         89.150427
2025-04-03  2025-04-04          84.75          81.46          0.000655         84.805489
2025-04-04  2025-04-10          81.46          79.91          0.000057         81.464654
2025-04-10  2025-04-11          79.91          79.84          0.000178         79.924233
2025-04-11  2025-04-14          79.84          80.98          0.000130         79.850378


## 3. Compare gasoil predictions with no change

The predicted price is `G_origin × (1 + predicted_return)`. Persistence predicts the origin price, equivalent to predicting zero return. MAE is average absolute error; RMSE places greater weight on large errors. Both below are USD/barrel.

A high price-level R² can simply reflect price persistence. The important first question is whether error is lower than the persistence baseline. Return-level R² is printed separately.


In [4]:
rows = []
for split, values in result["gasoil_metrics"].items():
    a, b = values["regression"], values["persistence"]
    rows.append({"split": split, "n": a["n"], "regression_MAE": a["mae"],
                 "baseline_MAE": b["mae"], "regression_RMSE": a["rmse"],
                 "baseline_RMSE": b["rmse"],
                 "return_R2": values["return_regression"]["r2"]})
print(pd.DataFrame(rows).to_string(index=False))


     split    n  regression_MAE  baseline_MAE  regression_RMSE  baseline_RMSE  return_R2
     train 1754        0.867400      0.865462         1.733594       1.731388   0.002401
validation  375        0.772214      0.761040         1.249153       1.239883  -0.016274
      test  376        1.942768      1.934548         4.336405       4.314289  -0.011840


## 4. Check the gasoil-to-MGO relationship separately

We pair adjacent **actual** MGO quotes only when gasoil values exist on both dates and the interval is at most 30 calendar days. Using actual gasoil changes, compare:

- Persistence: `M_future = M_origin`.
- Percentage transfer: `M_future = M_origin × (1 + actual_gasoil_return)`.
- Fitted relationship: `MGO_return = alpha + beta × actual_gasoil_return`, calibrated on training-period intervals only.

**This diagnostic uses realized gasoil, not predicted gasoil. It cannot establish forecasting accuracy.** Intervals vary in duration, so the fitted relationship is not applied to a one-step forecast. There are too few matching daily training outcomes to establish a calibrated daily mapping.


In [5]:
rows = []
for split, values in result["mapping_diagnostic_metrics"].items():
    row = {"split": split, "n": values["persistence"]["n"],
           "min_days": values["interval_days_min"], "max_days": values["interval_days_max"],
           "transfer_MAE": values["actual_gasoil_transfer"]["mae"],
           "baseline_MAE": values["persistence"]["mae"]}
    if "calibrated_actual_gasoil" in values:
        row["calibrated_MAE"] = values["calibrated_actual_gasoil"]["mae"]
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))
print("\nMixed-interval diagnostic regression:")
print(json.dumps(result["mapping_model"], indent=2))


     split  n  min_days  max_days  transfer_MAE  baseline_MAE  calibrated_MAE
     train 30         1        28     22.949208     39.600000       25.530595
validation  7         1        29     16.273013     17.785714       16.931579
      test 77         1        24     30.951805     31.136364       31.457323

Mixed-interval diagnostic regression:
{
  "intercept": 0.011081412255820142,
  "slopes": [
    0.8070215320249617
  ],
  "rank": 2,
  "training_rows": 30
}


## 5. Evaluate the complete pipeline

Now use **predicted** gasoil returns:

`MGO_predicted = observed_MGO_origin × (1 + predicted_gasoil_return)`

Only dates with actual MGO quotes at both gasoil forecast origin and target qualify. No synthetic target prices are made. The horizon is one gasoil trading step, which may span multiple calendar days. These matched observations are a selective subset; inspect their dates and quote provenance before generalizing. Errors below are USD/metric tonne.


In [6]:
rows = []
for split, values in result["end_to_end_mgo_metrics"].items():
    a, b = values["transfer"], values["persistence"]
    rows.append({"split": split, "n": a["n"], "pipeline_MAE": a["mae"],
                 "baseline_MAE": b["mae"], "pipeline_RMSE": a["rmse"],
                 "baseline_RMSE": b["rmse"]})
print(pd.DataFrame(rows).to_string(index=False))


     split  n  pipeline_MAE  baseline_MAE  pipeline_RMSE  baseline_RMSE
     train  7     15.376238     15.500000      20.478934      20.524376
validation  1      5.678818      6.000000       5.678818       6.000000
      test 82     25.461046     25.432927      42.536445      42.730901


## 6. Conclusion and what to tell the team

In this first run, the gasoil regression does not beat persistence on test MAE. The MGO percentage-transfer pipeline also has slightly worse test MAE than persistence. This is a valid result: the initial model has not demonstrated useful forecasting improvement.

The purpose of this notebook is to make the modelling approach reproducible and assessable. It does not prove that all linear regression specifications fail. Do not tune to these test results and then reuse the same test set as independent final evidence.

Next, confirm the forecast horizon, investigate source timing and quote provenance, and evaluate a predeclared next experiment using expanding time windows. Mapping uncertainty, anchor freshness, futures roll treatment and historical revisions still need investigation. Full-day news cannot simply be joined as if it were available earlier that day; monthly sales require release availability rules.

For submission, explain the target, four predictors, chronological split, benchmark, two-stage mapping assumption and failure to outperform the benchmark. The CSVs provide every prediction and the report gives a readable handover.


In [7]:
print("Report:", OUTPUT_DIR / "report.md")
print("Full numerical results:", OUTPUT_DIR / "results.json")
print("Gasoil predictions:", OUTPUT_DIR / "gasoil_predictions.csv")
print("MGO predictions:", OUTPUT_DIR / "mgo_predictions.csv")


Report: /Users/nelsontan/Desktop/Y2S1/CS203/cs203-g2t5/linear regression/outputs/report.md
Full numerical results: /Users/nelsontan/Desktop/Y2S1/CS203/cs203-g2t5/linear regression/outputs/results.json
Gasoil predictions: /Users/nelsontan/Desktop/Y2S1/CS203/cs203-g2t5/linear regression/outputs/gasoil_predictions.csv
MGO predictions: /Users/nelsontan/Desktop/Y2S1/CS203/cs203-g2t5/linear regression/outputs/mgo_predictions.csv
